In [3]:
import numpy as np
import pandas as pd
import optuna
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# 1. Simulasi Dataset train_transaction.csv (Biar Aman & Ringan di Laptop)
np.random.seed(42)
n_samples = 5000
features = {f'feature_{i}': np.random.randn(n_samples) for i in range(15)}
features['TransactionAmt'] = np.random.exponential(scale=100, size=n_samples)
df = pd.DataFrame(features)
# Mengatasi Class Imbalance (Fraud biasanya hanya ~3% dari total data)
df['isFraud'] = np.random.choice([0, 1], size=n_samples, p=[0.97, 0.03])

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data Preprocessing Soal 1 Selesai!")

Data Preprocessing Soal 1 Selesai!


In [4]:
# Jangan aktifkan autolog (menghindari konflik param); gunakan parent run dan nested runs

from sklearn.neural_network import MLPClassifier

def objective(trial):
    # Parameter yang akan di-tune otomatis oleh Optuna
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    num_units = trial.suggest_categorical('num_units', [16, 32, 64])
    alpha = trial.suggest_float('alpha', 1e-5, 1e-1, log=True)

    with mlflow.start_run(nested=True):
        hidden_layers = (num_units,)
        clf = MLPClassifier(hidden_layer_sizes=hidden_layers, alpha=alpha, learning_rate_init=lr, max_iter=200, random_state=42)
        clf.fit(X_train_scaled, y_train)
        preds = clf.predict_proba(X_test_scaled)[:, 1]
        score = roc_auc_score(y_test, preds)

        # Log parameter khusus ke MLflow (autolog juga mencatat otomatis)
        mlflow.log_param("num_units", int(num_units))
        mlflow.log_param("alpha", float(alpha))
        mlflow.log_param("learning_rate", float(lr))
        mlflow.log_metric("roc_auc_score", float(score))

    return score

# Jalankan Studi Optuna sebanyak 5 kali percobaan
mlflow.set_experiment("UAS_Fraud_Detection")
# Buat parent run supaya setiap trial dapat membuat nested run tanpa bentrok param
with mlflow.start_run():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=5)

print("\n--- TUNING SELESAI ---")
print("Parameter Terbaik:", study.best_params)
print("ROC-AUC Terbaik:", study.best_value)

[I 2026-06-23 22:30:51,472] A new study created in memory with name: no-name-1e34977d-7d34-4fab-bec9-2dae2fbe2079
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2026-06-23 22:30:52,320] Trial 0 finished with value: 0.4541446208112875 and parameters: {'learning_rate': 0.008347780182363485, 'num_units': 32, 'alpha': 1.3569537713879809e-05}. Best is trial 0 with value: 0.4541446208112875.
c:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2026-06-23 22:30:53,423] Trial 1 finished with value: 0.4506907701352146 and parameters: {'learning_rate': 0.000285818373862562, 'num_units': 64, 'alpha': 0.06661952798390773}. Best is trial 0 with value: 0.45414


--- TUNING SELESAI ---
Parameter Terbaik: {'learning_rate': 0.0001194207598945046, 'num_units': 32, 'alpha': 1.421412031238031e-05}
ROC-AUC Terbaik: 0.4880952380952381
